# Phase 1: Colab Fine-Tuning Preparation for LFM2.5-VL-450M

This script is intended to be run on Google Colab (T4 GPU). It pulls historical satellite imagery from the SimSat API, loads the LiquidAI model, sets up LoRA fine-tuning for wildfire smoke/thermal anomaly detection, and exports the adapter weights.

In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
# This will prompt you for authorization
drive.mount('/content/drive')

# 2. Define paths (Adjust if your zip is in a subfolder)
DRIVE_ZIP_PATH = '/content/drive/MyDrive/SimSatHackathon/geofire_dataset.zip'
LOCAL_EXTRACT_PATH = './geofire_data/'

# 3. Extract the dataset
if os.path.exists(DRIVE_ZIP_PATH):
    !unzip -q {DRIVE_ZIP_PATH} -d {LOCAL_EXTRACT_PATH}
    print("✅ Dataset extracted from Drive.")
else:
    print(f"❌ Error: Could not find {DRIVE_ZIP_PATH}. Please check your Drive path.")

# 4. Install ML libraries
!pip install -q transformers datasets accelerate peft bitsandbytes pillow torch torchvision

Mounted at /content/drive
✅ Dataset extracted from Drive.
ERROR: Operation cancelled by user


In [ ]:
import torch
from PIL import Image
from datasets import load_dataset
# Updated: AutoModelForImageTextToText is the new v5.0+ standard
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import LoraConfig, get_peft_model
import os

MODEL_ID = "LiquidAI/LFM2.5-VL-450M"
DATASET_JSON = "./geofire_data/lfm_finetune_dataset.json"
BASE_DIR = "./geofire_data"

print("Loading Multimodal Model (v5 API)...")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16
)

print("Applying Multimodal LoRA...")
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    # Task type for Image-to-Text generation
    task_type="FEATURE_EXTRACTION" if not hasattr(model, "lm_head") else "CAUSAL_LM"
)
# Note: If CAUSAL_LM fails, you can set task_type=None and let PEFT auto-detect
model = get_peft_model(model, lora_config)

def format_vlm_data(example):
    image_path = os.path.join(BASE_DIR, example["image"])
    image = Image.open(image_path).convert("RGB")
    user_prompt = example["conversations"][0]["value"]
    assistant_response = example["conversations"][1]["value"]

    # Standard Liquid prompt format
    full_prompt = f"<image>\nUSER: {user_prompt}\nASSISTANT: {assistant_response}"

    inputs = processor(text=full_prompt, images=image, return_tensors="pt",
                       padding="max_length", max_length=512, truncation=True)
    inputs = {k: v.squeeze(0) for k, v in inputs.items()}
    inputs["labels"] = inputs["input_ids"].clone()
    return inputs

raw_dataset = load_dataset("json", data_files=DATASET_JSON, split="train")
processed_dataset = raw_dataset.map(format_vlm_data, remove_columns=raw_dataset.column_names)

In [ ]:
from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer
from google.colab import runtime

DRIVE_SAVE_PATH = '/content/drive/MyDrive/SimSatHackathon/geofire_orbital_weights'

print("Initializing Training...")
training_args = TrainingArguments(
    output_dir='./local_output',
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=10,
    logging_steps=1,
    save_strategy="no",
    fp16=True,
    optim="paged_adamw_8bit",
    remove_unused_columns=False # Important: Vision-Language models need all input columns
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer=processor.tokenizer, mlm=False),
)

trainer.train()

# Save and backup to Drive
trainer.model.save_pretrained('./final_adapter')
processor.save_pretrained('./final_adapter')

print(f"Moving weights to Google Drive...")
if not os.path.exists(DRIVE_SAVE_PATH):
    os.makedirs(DRIVE_SAVE_PATH)

!cp -r ./final_adapter/* {DRIVE_SAVE_PATH}
print("✅ Weights successfully backed up to Drive.")

runtime.unassign()